参考链接：
https://zhuanlan.zhihu.com/p/23824881486

# Task：无状态分布式任务

In [1]:
import ray

ray.init(num_cpus=2)


# 使用 @ray.remote 装饰器定义分布式任务
@ray.remote
def add(x, y):
    return x + y


# 异步提交任务，立即返回 future 对象（对象引用）
future1 = add.remote(1, 2)
future2 = add.remote(3, 4)

# 等待任务完成并获取结果
results = ray.get([future1, future2])
print(results)  # 输出: [3, 7]

# 关闭 Ray
ray.shutdown()


2025-11-03 15:30:00,504	INFO worker.py:2013 -- Started a local Ray instance.
/data/ljr/anaconda3/envs/dpspeed/lib/python3.10/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


[3, 7]


# Actor：有状态的分布式对象
与无状态的 Task 不同，Ray Actor 提供了一种面向对象的分布式编程模型，能够维护状态并封装方法。它特别适合需要保持状态的场景，如参数服务器、计数器等。

下面通过一个计数器示例来展示 Actor 的基本用法：

In [2]:
import ray

ray.init()

@ray.remote
class Counter:
    def __init__(self):
        self.value = 0
    
    def increment(self):
        self.value +=1
        return self.value

    def get_value(self):
        return self.value

# 创建 Actor 实例（在远程工作进程中）
counter = Counter.remote()

# 异步调用 Actor 方法
future1 = counter.increment.remote()  # 第一次增加
future2 = counter.increment.remote()  # 第二次增加
future3 = counter.get_value.remote()  # 获取当前值

# 获取结果
print(ray.get([future1, future2]))  # 输出: [1, 2]
print(ray.get(future3))  # 输出: 2

ray.shutdown()

2025-11-03 15:34:02,188	INFO worker.py:2013 -- Started a local Ray instance.


[1, 2]
2


Actor 的特性：

状态持久化：Actor 实例在其生命周期内可以维持状态

串行执行：同一个 Actor 的方法调用按顺序执行，保证状态一致性

并发调用：不同 Actor 实例之间可以并行执行

远程通信：通过 .remote() 进行异步方法调用

与 Task 不同，Actor 更适合需要维护状态的长期运行的计算任务，如模型训练、参数更新等场景。

# Ray 常用操作指南
本节介绍 Ray 中最常用的操作，这些是构建分布式应用的基础。

## 数据操作：ray.put() 和 ray.get()

In [3]:
import ray
ray.init()

# ray.put(): 将对象存入共享内存
data = [1, 2, 3, 4, 5]
data_ref = ray.put(data)  # 返回对象引用

# ray.get(): 从共享内存获取对象
result = ray.get(data_ref)  # [1, 2, 3, 4, 5]

2025-11-03 15:36:13,273	INFO worker.py:2013 -- Started a local Ray instance.


## 任务管理：ray.wait()

用于监控和等待异步任务的完成状态：

In [7]:
import time

# 等1 2 3 4秒
@ray.remote
def slow_func(i):
    time.sleep(i+1)
    return i 

# 提交多个异步任务
refs= [slow_func.remote(i) for i in range(4)]

# 等待部分任务完成
start_time = time.time()
ready_refs, remaining_refs = ray.wait(refs, num_returns=2)  
wait_time = time.time() - start_time
print(f"等待时间：{wait_time}，完成: {len(ready_refs)}, 等待中: {len(remaining_refs)}")  # 完成: 2, 等待中: 2

start_time = time.time()
# 等待所有任务完成
# 从此刻起：

# 距第 2 个任务（sleep(2)）完成还需约 1 秒（因为它在第 4 秒时结束）；
# 距第 3 个任务（sleep(3)）完成还需约 2 秒（在第 4 秒时结束）。

ready_refs, remaining_refs = ray.wait(refs, num_returns=len(refs))
wait_time = time.time() - start_time
print(f"等待时间：{wait_time}，完成: {len(ready_refs)}, 等待中: {len(remaining_refs)}")  # 完成: 4, 等待中: 0
print("结果:", ray.get(ready_refs))  # 结果: [0, 1, 2, 3]

等待时间：2.0187883377075195，完成: 2, 等待中: 2
等待时间：2.0100157260894775，完成: 4, 等待中: 0
结果: [0, 1, 2, 3]


## Actor 管理：ray.kill()

In [10]:
@ray.remote
class Counter1:
    def __init__(self):
        self.value = 0
    def increment(self):
        self.value += 1
        return self.value

# 创建并终止 Actor
cnt = Counter1.remote()
ray.kill(cnt)


# 后续调用将抛出异常
try:
    result = ray.get(counter.increment.remote())
except ray.exceptions.RayActorError as e:
    print("Actor 已终止:", e)

Exception: Failed to submit task to actor ActorID(2d523c8359ad4e99a03085f701000000) due to b"Can't find actor 2d523c8359ad4e99a03085f701000000. It might be dead or it's from a different cluster"

## 资源管理
Ray 支持灵活的计算资源分配：

In [11]:
# 静态资源配置
@ray.remote(num_cpus=2)
def cpu_task():
    return 1

@ray.remote(num_gpus=1)
def gpu_task():
    return 1

# 动态资源配置
@ray.remote
def task():
    return 1

future = task.options(num_cpus=2, num_gpus=1).remote()

## 错误处理与可靠性
提供全面的异常处理和重试机制：

In [15]:
# 异常处理
@ray.remote
def might_fail(x):
    if x < 0:
        raise ValueError("不允许负值")
    return x

try:
    result = ray.get(might_fail.remote(-1))
except ray.exceptions.RayTaskError as e:
    print("任务执行失败:", e)

# 自动重试机制
@ray.remote(max_retries=3)
def unstable_function():
    # 可能失败的操作
    ray.get(might_fail.remote(-1))
    pass

任务执行失败: ray::might_fail() (pid=2373080, ip=10.106.17.252)
  File "/tmp/ipykernel_2363635/1601044204.py", line 5, in might_fail
ValueError: 不允许负值
